In [3]:
#Cell 1 — setup (same pattern as before, reusing your existing checkpoint):

!pip install decord

import os
import json
from typing import List, Dict

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from decord import VideoReader, cpu

from google.colab import drive
drive.mount('/content/drive')

MSVD_ROOT = "/content/drive/Shareddrives/DATA 298A/DATA/MSVD"
CHECKPOINT = "/content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip/best.pt"
MODEL_NAME = "openai/clip-vit-base-patch32"

print("MSVD_ROOT exists:", os.path.exists(MSVD_ROOT))
print("CHECKPOINT exists:", os.path.exists(CHECKPOINT))

Mounted at /content/drive
MSVD_ROOT exists: True
CHECKPOINT exists: True


In [6]:
#Cell 2 — load the model and checkpoint:
def extract_tensor_features(output, kind="image"):
    if torch.is_tensor(output):
        return output
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    if isinstance(output, (tuple, list)) and len(output) > 0:
        if torch.is_tensor(output[0]):
            return output[0]
    raise TypeError(f"Could not extract tensor features from {type(output)}")


class CLIPVideoTextModel(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_text(self, input_ids, attention_mask):
        try:
            text_features = self.clip.get_text_features(
                input_ids=input_ids, attention_mask=attention_mask
            )
        except Exception:
            out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
            text_features = extract_tensor_features(out, kind="text")
        text_features = extract_tensor_features(text_features, kind="text")
        return F.normalize(text_features, dim=-1)

    def encode_frames(self, pixel_values):
        try:
            frame_features = self.clip.get_image_features(pixel_values=pixel_values)
        except Exception:
            out = self.clip.vision_model(pixel_values=pixel_values)
            frame_features = extract_tensor_features(out, kind="image")
        frame_features = extract_tensor_features(frame_features, kind="image")
        return F.normalize(frame_features, dim=-1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPVideoTextModel(MODEL_NAME).to(device)
load_checkpoint(model, CHECKPOINT)
model.eval();

Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded checkpoint from: /content/drive/Shareddrives/DATA 298A/Models/CLIP/output_msvd_clip/best.pt


In [7]:
#Cell 3 — the core timestamp function:
def get_event_timestamp(video_path: str, query: str, model, processor, device, num_frames: int = 8):
    """
    Given a video and a text query, returns the predicted timestamp (in seconds)
    of the frame that best matches the query, plus its similarity score.
    """
    vr = VideoReader(video_path, ctx=cpu(0))
    total_frames = len(vr)
    fps = vr.get_avg_fps()

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = vr.get_batch(frame_indices).asnumpy()
    pil_frames = [Image.fromarray(f).convert("RGB") for f in frames]

    image_inputs = processor(images=pil_frames, return_tensors="pt").to(device)
    text_inputs = processor(text=[query], return_tensors="pt", padding=True, truncation=True, max_length=32).to(device)

    with torch.no_grad():
        frame_embeds = model.encode_frames(image_inputs["pixel_values"])   # [num_frames, 512]
        text_embed = model.encode_text(text_inputs["input_ids"], text_inputs["attention_mask"])  # [1, 512]

        similarities = (frame_embeds @ text_embed.T).squeeze(-1).cpu().numpy()  # [num_frames]

    best_idx_in_sample = int(np.argmax(similarities))
    best_frame_number = int(frame_indices[best_idx_in_sample])
    predicted_timestamp_sec = best_frame_number / fps if fps > 0 else None

    return {
        "predicted_timestamp_sec": predicted_timestamp_sec,
        "best_frame_number": best_frame_number,
        "total_frames": total_frames,
        "fps": float(fps),
        "similarity_score": float(similarities[best_idx_in_sample]),
        "per_frame_similarities": similarities.tolist(),
    }


# Quick sanity test on one known video/query pair before running on more
test_video_path = os.path.join(MSVD_ROOT, "raw_videos", "fr9H1WLcF1A_256_261.avi")
test_query = "two young men are playing table tennis"

result = get_event_timestamp(test_video_path, test_query, model, processor, device)
print("Test result:")
for k, v in result.items():
    if k != "per_frame_similarities":
        print(f"  {k}: {v}")

Test result:
  predicted_timestamp_sec: 4.971633333333333
  best_frame_number: 149
  total_frames: 150
  fps: 29.97002997002997
  similarity_score: 0.325438916683197


In [8]:
#Cell 4 — run on a handful more samples to build real evidence for the PR
msvd_test_path = os.path.join(MSVD_ROOT, "msvd_test.json")
with open(msvd_test_path, "r", encoding="utf-8") as f:
    msvd_test_data = json.load(f)

# Pick 5 diverse samples to demonstrate the function works across different videos
sample_records = msvd_test_data[:5]

demo_results = []
for record in sample_records:
    video_path = os.path.join(MSVD_ROOT, "raw_videos", record["video"])
    if not os.path.exists(video_path):
        continue

    caption = record["caption"][0] if isinstance(record["caption"], list) else record["caption"]

    result = get_event_timestamp(video_path, caption, model, processor, device)
    demo_results.append({
        "video_id": record["video_id"],
        "query": caption,
        "predicted_timestamp_sec": result["predicted_timestamp_sec"],
        "total_duration_sec": result["total_frames"] / result["fps"] if result["fps"] > 0 else None,
        "similarity_score": result["similarity_score"],
    })

    print(f"Video: {record['video_id']}")
    print(f"  Query: {caption}")
    print(f"  Predicted timestamp: {result['predicted_timestamp_sec']:.2f}s (clip duration: {result['total_frames']/result['fps']:.2f}s)")
    print(f"  Similarity: {result['similarity_score']:.3f}")
    print()

# Save as evidence
OUTPUT_JSON = "/content/clip_timestamp_demo_results.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(demo_results, f, indent=2)
print(f"Saved to {OUTPUT_JSON}")

Video: fr9H1WLcF1A_256_261
  Query: two young men are playing table tennis
  Predicted timestamp: 4.97s (clip duration: 5.00s)
  Similarity: 0.325

Video: wFPmKChNrhU_3_11
  Query: a man rides his horse in the dessert
  Predicted timestamp: 7.97s (clip duration: 8.01s)
  Similarity: 0.318

Video: o4OsYxsNGMI_77_82
  Query: paper is being cut with scissors
  Predicted timestamp: 0.00s (clip duration: 5.04s)
  Similarity: 0.291

Video: rw9h_574HxE_59_66
  Query: a couple walks through the jungle
  Predicted timestamp: 0.00s (clip duration: 7.04s)
  Similarity: 0.286

Video: jDFn-1lXJ98_71_80
  Query: a dog runs and hides behind a tree
  Predicted timestamp: 5.11s (clip duration: 9.01s)
  Similarity: 0.269

Saved to /content/clip_timestamp_demo_results.json
